In [35]:
import os
import openai

openai.api_key = os.environ["OPENAI_API_KEY"]

def llm(prompt, stop=["\n"]):
    response = openai.Completion.create(
      model="text-davinci-002",
      prompt=prompt,
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    )
    return response["choices"][0]["text"]

def completion(model, messages, temperature=0, max_tokens=100, top_p=1, stop=['\n']):
    response = openai.ChatCompletion.create(
      model=model,
      messages=messages,
      max_tokens=max_tokens,
      stop=stop,
      temperature=temperature,
      top_p=top_p,
      frequency_penalty=0.0,
      presence_penalty=0.0,
    )
    msg = response.choices[0]["message"]
    action = msg["content"].strip()
    if not action and "reasoning_content" in msg:
        reason = msg["reasoning_content"].strip()
        if reason:
            action = "think: " + reason
    return action

In [36]:
import yaml
import alfworld
import alfworld.agents.environment
with open('base_config.yaml') as reader:
    config = yaml.safe_load(reader)
    
split = "eval_out_of_distribution"

env = getattr(alfworld.agents.environment, config["env"]["type"])(config, train_eval=split)
env = env.init_env(batch_size=1)

def process_ob(ob):
    if ob.startswith('You arrive at loc '):
        ob = ob[ob.find('. ')+2:]    
    return ob

Initializing AlfredTWEnv...
Checking for solvable games...


100%|██████████| 341/341 [00:00<00:00, 1980.55it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games


In [37]:
import json
folder = './prompts/'
prompt_file = 'alfworld_3prompts.json'
with open(folder + prompt_file, 'r') as f:
    d = json.load(f)

In [38]:
import sys

def create_messages(prompt, ob):
    history = [{
        'role': 'system',
        'content': prompt[0]        
    }]

    def shot_to_history(shot) -> None:
        cnt = 0
        for s in shot.split('\n'):
            msg = s.strip()
            if not msg:
                continue
            if msg.startswith('>'):
                msg = msg.removeprefix('>').lstrip()
                assert msg
                assert cnt > 0 and (history[-1]['role'] != 'assistant' or history[-1]['content'].startswith('think:'))
                history.append({
                    'role': 'assistant',
                    'content': msg
                })
            else:
                if msg == 'OK.':
                    assert cnt > 0 and history[-1]['role'] == 'assistant' and history[-1]['content'].startswith('think:')
                elif cnt > 0 and history[-1]['role'] == 'user':
                    history[-1]['content'] += '\n' + msg
                    continue
                if cnt == 0:
                    msg = "[New Task] " + msg
                history.append({
                    'role': 'user',
                    'content': msg
                })
            cnt += 1

    for shot in prompt[1:]:
        shot_to_history(shot)
    history.append({
        'role': 'user',
        'content': '[New Task] ' + ob
    })
    return history

def alfworld_run(prompt, to_print=True, ob=''):
    #init_prompt = prompt + ob + '\n>'
    #prompt = ''
    if to_print:
        print(ob)
        sys.stdout.flush()
    messages = create_messages(prompt, ob)
    for i in range(1, 50):
        action = completion(model="qwen3.5-9b", messages=messages, max_tokens=256, stop=[]) #llm(init_prompt + prompt, stop=['\n']).strip()
        observation, reward, done, info = env.step([action])
        observation, reward, done = process_ob(observation[0]), info['won'][0], done[0]
        if action.startswith('think:'):
            observation = 'OK.'
        if to_print:
            print(f'Act {i}: {action}\nObs {i}: {observation}')
            sys.stdout.flush()
        #prompt += f' {action}\n{observation}\n>'
        messages.append({
            'role': 'assistant',
            'content': action
        })
        messages.append({
            'role': 'user',
            'content': observation
        })
        if done:
            return reward
    return 0

In [39]:
prefixes = {
    'pick_and_place': 'put',
    'pick_clean_then_place': 'clean',
    'pick_heat_then_place': 'heat',
    'pick_cool_then_place': 'cool',
    'look_at_obj': 'examine',
    'pick_two_obj': 'puttwo'
}
cnts = [0] * 6
rs = [0] * 6
n = 1 #134

for _ in range(n):
    ob, info = env.reset()
    ob = '\n'.join(ob[0].split('\n\n')[1:])
    name = '/'.join(info['extra.gamefile'][0].split('/')[-3:-1])
    print(name)
    for i, (k, v) in enumerate(prefixes.items()):
        if name.startswith(k):
            prompt = ['Interact with a household to solve a task.', # Here are two examples.\n' + d[f'react_{v}_1'] + d[f'react_{v}_0'] + '\nHere is the task.\n'
                d[f'react_{v}_1'],
                d[f'react_{v}_0']
            ]
            print(k, v)
            r = alfworld_run(prompt, ob=ob)
            rs[i] += r
            cnts[i] += 1
            break
    print(_+1, 'r', r, 'rs', rs, 'cnts', cnts, 'sum(rs)/sum(cnts)', sum(rs) / sum(cnts))
    print('------------\n')


pick_clean_then_place_in_recep-Pan-None-CounterTop-10/trial_T20190908_032434_013084
pick_clean_then_place clean
You are in the middle of a room. Looking quickly around you, you see a cabinet 6, a cabinet 5, a cabinet 4, a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 3, a countertop 2, a countertop 1, a drawer 3, a drawer 2, a drawer 1, a fridge 1, a garbagecan 1, a microwave 1, a shelf 3, a shelf 2, a shelf 1, a sinkbasin 1, a stoveburner 4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.
Your task is to: clean some pan and put it in countertop.
Act 1: think: To solve the task, I need to find and take a pan, then clean it with sinkbasin, then put it in countertop. First, I need to find a pan. Pans are often found in cabinets or drawers. I will check cabinet
Obs 1: OK.
Act 2: go to cabinet 1
Obs 2: On the cabinet 1, you see a glassbottle 1, a mug 1, and a vase 2.
Act 3: go to cabinet 2
Obs 3: The cabinet 2 is closed.
Act 4: go to cabinet 3
Obs